# Practice: Trace a Story Across a Corpus

**Computational Text Analysis Pack · Module 18 — Text Reuse and Intertextuality**

*Technique:* Detection, filtering, clustering and the direction problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/08-text-reuse-and-networks/18d_practice_reuse_study.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **Which stories in this corpus travelled — appearing in more than one outlet
> or language — and can I say which version came first?**

The first half is what reuse detection does well. The second half is where most
reuse studies overreach, and this notebook is careful about the difference.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
from collections import defaultdict

def shingles(text, k=4):
    toks = tokenise(str(text))
    return {" ".join(toks[i:i+k]) for i in range(len(toks)-k+1)}
def jaccard(a, b):
    return len(a & b)/len(a | b) if a and b else 0.0

docs = df.sample(n=500, random_state=11).reset_index(drop=True)
sets = [shingles(t) for t in docs["text"]]
print(f"{len(docs)} documents shingled")

# Step 1: strip boilerplate before believing anything (module 18c's warning).
sh_count = Counter()
for s in sets: sh_count.update(s)
boiler = {sh for sh, n in sh_count.items() if n > len(sets)*0.05}
print(f"{len(boiler)} shingles appear in >5% of documents — treated as boilerplate")
sets = [s - boiler for s in sets]

In [ ]:
# Step 2: detect.
pairs = []
for i in range(len(sets)):
    for j in range(i+1, len(sets)):
        s = jaccard(sets[i], sets[j])
        if s > 0.08:
            pairs.append((s, i, j))
pairs.sort(reverse=True)
print(f"{len(pairs)} candidate reuse pairs above 0.08\n")
for s, i, j in pairs[:5]:
    print(f"[{s:.3f}] {docs.loc[i,'language']:<16}{str(docs.loc[i,'headline'])[:52]}")
    print(f"         {docs.loc[j,'language']:<16}{str(docs.loc[j,'headline'])[:52]}\n")

In [ ]:
# Step 3: cluster into stories, not pairs. A story may have five versions.
parent = list(range(len(sets)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

for s, i, j in pairs:
    union(i, j)
clusters = defaultdict(list)
for i in range(len(sets)):
    clusters[find(i)].append(i)
multi = {k: v for k, v in clusters.items() if len(v) > 1}
print(f"{len(multi)} clusters containing more than one document")
sizes = Counter(len(v) for v in multi.values())
print("cluster sizes:", dict(sorted(sizes.items())))

biggest = max(multi.values(), key=len)
print(f"\nlargest cluster ({len(biggest)} documents):")
for i in biggest[:6]:
    print(f"   ({docs.loc[i,'language']}) {str(docs.loc[i,'headline'])[:66]}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ks = sorted(sizes)
ax.bar([str(k) for k in ks], [sizes[k] for k in ks], color="steelblue")
ax.set_xlabel("documents in cluster"); ax.set_ylabel("clusters")
ax.set_title("Most reuse is pairwise; a few stories travel widely")
plt.tight_layout()

langs = Counter()
for v in multi.values():
    if len(set(docs.loc[i, "language"] for i in v)) > 1:
        langs["cross-language"] += 1
    else:
        langs["same-language"] += 1
print(dict(langs))

## The direction problem

Here is where reuse studies go wrong. You now have clusters. It is tempting to
say version A is the source and B the copy.

**You cannot get that from similarity.** The graph is undirected. Deciding
priority needs evidence the method does not contain:

- **Dates**, which this corpus does not carry.
- **External knowledge** of which outlet is a wire service and which subscribes.
- **Textual evidence** — an error in one version that the other reproduces is
  the classic philological argument, and it is stronger than any similarity
  score.

**Defensible:** "N clusters of related documents were detected at Jaccard ≥
0.08 after boilerplate removal; M of them span more than one language."

**Not defensible:** "Outlet X copied outlet Y." That claim needs a timeline.

### 🔧 Your turn 1

Raise the threshold to 0.2 and re-run. How many clusters survive, and are the survivors more convincing when you read them?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Far fewer clusters survive and they are usually near-duplicates — which is
either exactly what you wanted or too strict, depending on whether you are
studying syndication (high threshold) or looser rewriting and adaptation (low).
Reporting the threshold is not optional, because the number of "reuse events"
in your corpus is a function of it, and two studies with different thresholds
are not comparable even on the same archive.

</details>